# Creación de un *CRUD* con `sqlite3` (intermedio)
En esta versión del código, haré uso de funciones para simplificar las tareas del CRUD.

Recomendación: instalar extensión [SQLite Viewer](https://marketplace.visualstudio.com/items?itemName=qwtel.sqlite-viewer)

In [26]:
import sqlite3
from pathlib import Path
import os
import json

## Establecer directorio de trabajo

In [27]:
PROJECT_HOME_FOLDER = '/home/daniel/code/my_python_course/sqlite_crud'

if os.getcwd() == PROJECT_HOME_FOLDER:
    os.chdir(PROJECT_HOME_FOLDER + "/data/")

In [28]:
# os.getcwd()

In [29]:
# db_files = [i for i in Path(".").iterdir() if i.suffix == ".db"]
# print(db_files)

## Funciones auxiliares

Crearé algunas funciones auxiliares que me permitan realizar algunas tareas que ayuden en las tareas de mantenimiento de la base de datos y sus tablas.

### Listado de tablas de la BD
Esta función devuelve una lista de las tablas que contiene la BD de la conexión activa.

In [30]:
def list_tables(conn, cursor):
    tables = (cursor
              .execute("SELECT name FROM sqlite_master WHERE type='table'")
              .fetchall()
    )
    tables = [i[0] for i in tables if i[0] != "sqlite_sequence"]
    return tables

## Iniciar conexión  

In [31]:
def init_connection(db_file : str) -> tuple[sqlite3.Connection, sqlite3.Cursor] | None :
    # check if file exists
    if (Path(".") / db_file).exists():
        try:
            conn = sqlite3.connect(db_file)
            cursor = conn.cursor()
            print("Connection created successfuly!")
            return conn, cursor
        except Exception as e:
            print(f"Error: {e}")
            return None
    else:
        print(f"{db_file} does not exist. I will create it.")
        try:
            conn = sqlite3.connect(db_file)
            cursor = conn.cursor()
            print("Connection created successfuly!")
            print(f"Database file created: {db_file}")
            return conn, cursor
        except Exception as e:
            print(f"Error: {e}")
            return None

In [32]:
conn, cursor = init_connection("inventario.db")

Connection created successfuly!


In [33]:
list_tables(conn, cursor)

['productos']

## C (create)

### Creación de tabla
Esta función crea una tabla a partir de una *query* con el código SQL de creación de la tabla.

In [34]:
def create_table(table_name, query, conn, cursor):
    try:
        # check if table exists
        if not table_name in list_tables(conn, cursor):
            cursor.execute(query)
            conn.commit()

            # get the database list
            db_details = conn.execute("PRAGMA database_list").fetchall()
            db_name = db_details[0][2].split("/")[-1]

            # success message
            print(f"Table {table_name} was successfuly created at database {db_name}!")
        
        # table was already created
        else:
            print(f"Table {table_name} already exists. No actions were taken.")
    except Exception as e:
        print(f"Error: {e}")

Crear tabla *productos*

In [35]:
# create_table_productos_query ='''
#     CREATE TABLE IF NOT EXISTS productos (
#         id              INTEGER PRIMARY KEY AUTOINCREMENT,
#         nombre          TEXT    NOT NULL,
#         marca           TEXT    NOT NULL,
#         categoria       TEXT    NOT NULL,
#         precio          REAL    NOT NULL,
#         stock           INTEGER NOT NULL,
#         especificaciones TEXT,
#         fecha_ingreso   TEXT
#     )
# '''
# create_table("productos", create_table_productos_query, conn, cursor)

### Inserción de datos

In [36]:
def populate_table(table, cols, values, data_structure="dict", conn=conn, cursor=cursor):
    
    try:
        # check if data_structure value is valid:
        if data_structure not in ["tuples", "dict"]:
            print("Invalid data_structure value: must be tuples or dict!")
            return None
        
        # create a dictionary containing column names as keys
        # and values contained in each tuple provided in values
        
        if data_structure != "dict":
            values = [{k:v for k,v in zip(cols, t)}
                    for t in values]
        
        # create placeholders
        placeholders = []
        for c in cols:
            placeholders.append(f":{c}")
        placeholders = ", ".join(placeholders)

        
        insert_query = f'''
        insert into {table} ({", ".join(cols)})
        values ({placeholders}) ;
        '''

        # insert records into db
        cursor.executemany(insert_query, values)
        conn.commit()
        
        # check if insertion was ok
        records_count_in_values = len(values)
        records_count_in_table= cursor.execute(f"select count(*) from {table} ;").fetchone()[0]

        if records_count_in_table == records_count_in_values:
            print(f"Insertion succesful. {records_count_in_table} records where inserted!")
        else:
            print("Something went wrong!")
            print(f"Count of provided values: {records_count_in_values}")
            print(f"Count of provided values: {records_count_in_table}")
            conn.rollback()
    
    except Exception as e:
        print(f"Error: {e}")

### Cargar los datos del inventario desde el archivo .json

In [37]:
with open("inventario.json", "r", encoding="utf-8") as file:
    productos = json.load(file)   

In [38]:
# productos[:2]

In [39]:
# list(productos[0].keys())

### Insertar datos de inventario de productos

In [40]:
# populate_table("productos", ['nombre', 'marca', 'categoria', 'precio',
#                               'stock', 'especificaciones', 'fecha_ingreso'], data_structure="dict", values = productos)

## R (read)

### Lectura de datos

In [41]:
def read_table(table, cols="*", filter = None, limit = None, cursor=cursor):
    try:
        if cols != "*":
            columns = ", ".join(cols)
        else:
            columns = "*" 
        query = f'''
        select {columns}
        from {table}
        '''
        if filter:
            query += f"\n\twhere {filter}"
        
        if limit:
            query += f"\nlimit {limit}"
        
        query += " ;"

        return cursor.execute(query).fetchall()
    
    except Exception as e:
        print(f"Error: {e}")

In [42]:
cursor.execute("select * from productos where categoria='Audio';").fetchall()

[(49,
  'Rode NT1 Signature',
  'Rode',
  'Audio',
  449.99,
  14,
  'Micrófono cardioide, 1 pulgada, XLR',
  '2024-02-26'),
 (50,
  'Blue Yeti USB',
  'Blue',
  'Audio',
  99.99,
  72,
  'Micrófono USB, 4 patrones, silenciador',
  '2024-02-27')]

In [43]:
read_table(table = "productos", cols = ["nombre"])

[('iPhone 15 Pro Max',),
 ('Samsung Galaxy S24 Ultra',),
 ('MacBook Pro 16 M3',),
 ('Dell XPS 13',),
 ('AirPods Pro (2da gen)',),
 ('Sony WH-1000XM5',),
 ('iPad Air 11',),
 ('Samsung Galaxy Tab S9',),
 ('Canon EOS R6',),
 ('GoPro Hero 12',),
 ('Samsung 55" QLED TV',),
 ('LG 65" OLED TV',),
 ('Nintendo Switch OLED',),
 ('PlayStation 5',),
 ('Anker PowerBank 26800mAh',),
 ('iPhone 15',),
 ('Google Pixel 8 Pro',),
 ('OnePlus 12',),
 ('Lenovo ThinkPad X1 Carbon',),
 ('ASUS VivoBook 15',),
 ('Samsung Galaxy Buds2 Pro',),
 ('Beats Studio Pro',),
 ('JBL Flip 6',),
 ('iPad Pro 12.9 M2',),
 ('Microsoft Surface Go 3',),
 ('Sony Alpha 6700',),
 ('Nikon Z6II',),
 ('DJI Mini 3 Pro',),
 ('LG 55" QNED99 TV',),
 ('TCL 55" 5-Series TV',),
 ('Xbox Series X',),
 ('Valve Steam Deck OLED',),
 ('Razer DeathAdder V3',),
 ('Corsair K95 Platinum XT',),
 ('Logitech G Pro X 2',),
 ('Samsung 870 QVO 4TB',),
 ('WD Black SN850X 2TB',),
 ('Seagate Barracuda 4TB',),
 ('Belkin BoostCharge Pro',),
 ('SCUF Controller Pr

In [44]:
read_table("productos", cols=["nombre", "precio"], limit=3)

[('iPhone 15 Pro Max', 1299.99),
 ('Samsung Galaxy S24 Ultra', 1199.99),
 ('MacBook Pro 16 M3', 2499.99)]

## U (update)

### Actualización de registros

In [45]:
def update_records(table, col, value, filter = "", cursor=cursor):
    try:
        query = f'''
        update {table}
        set {col} = {value}
        '''
        if filter:
            query += "\n\twhere {filter}"
        
        query += "\n ;"

        cursor.execute(query)
        conn.commit()
        
    except Exception as e:
        print(f"Error: {e}")

## D (delete)

### Eliminación

#### Vaciado de la tabla

In [46]:
def truncate_table(table, conn=conn, cursor=cursor):
    try:
        cursor.execute(f"delete from {table} ;")
        
        # reset the id values
        cursor.execute(f'delete from sqlite_sequence where name ="{table}" ;')
        conn.commit()
        print(f"Table \"{table}\" has been cleaned from all records.")
        
    except Exception as e:
        print(f"Error: {e}")
    

In [47]:
# truncate_table("productos")

#### Eliminación de la tabla

In [48]:
def drop_table(table, cursor=cursor):
    try:
        cursor.execute(f"drop table {table} ;")

        # check if table has been removed from db
        tables_list = cursor.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
        tables_list = [i[0] for i in tables_list]
        
        if table not in tables_list:
            print(f"Table \"{table}\" has been removed from the database!")
        else:
            print(f'Table "{table}" could not be removed from database.')
            
    except Exception as e:
        print(f"Error: {e}")

#### Eliminación de registros

In [49]:
def delete_records(table, filter = "", cursor=cursor):
    try:
        if not filter:
            print("A filter is needed to perform selective deletion of records.")
            return None
        
        query = f"delete from {table} where {filter} ;"
        
        cursor.execute(query)
    
    except Exception as e:
        print(f"Error: {e}")

In [50]:
read_table("productos", cols=["nombre", "precio"], limit=10)

[('iPhone 15 Pro Max', 1299.99),
 ('Samsung Galaxy S24 Ultra', 1199.99),
 ('MacBook Pro 16 M3', 2499.99),
 ('Dell XPS 13', 1299.99),
 ('AirPods Pro (2da gen)', 249.99),
 ('Sony WH-1000XM5', 399.99),
 ('iPad Air 11', 799.99),
 ('Samsung Galaxy Tab S9', 549.99),
 ('Canon EOS R6', 2499.99),
 ('GoPro Hero 12', 499.99)]